<a href="https://colab.research.google.com/github/norasaleh1/Data-Science-Project/blob/main/Notebooks/Phase-3/Baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Here we installed rank-bm25 to use for the baseline model. We chose it because it searches through the data to find keyword matches.

In [13]:
pip install rank-bm25

In [14]:
#required imports:
import re
import string
import os
import pandas as pd
from sklearn.model_selection import train_test_split
import json
from rank_bm25 import BM25Okapi

1. We used **Gemini 2.5 Pro** to generate the `labor_law_qa.csv` file which contains 100 questions each generated from: `boe_cleaned.csv`,`istitlaa_cleaned.csv`, `qiwa_data_cleaned.csv`, and the whole `labor_law_faq_cleaned.csv` file added to them which in total gave us 321 samples.

2. We split the `labor_law_qa.csv` file into 321 QA pairs into 256 samples for training (80%) and 65 samples for testing (20%). We chose this split because a 90/10 would risk overfitting and 70/30 we wouldn't have enough data for training since our sample size is small (321 samples).

3. We used the **Source** column to ensure the 65 test questions are a good mix from all four data files, so the final exam balanced instead of being biased.

4. For saving the `train_data.jsonl`, `test_data.jsonl` files we made sure to write it in JSONL specific format (role: user/assistant/system) which is the standard for fine-tuning a LLM like Llama 3 which we will use for model C.

5. Lastly, we saved the same questions from the test set to the `test_set_ground_truth.csv` file with their correct answers so later we can grade all 3 models.

In [15]:
system_prompt = "You are an expert Saudi Labor Law consultant. Answer questions accurately based on the provided context."

def format_to_jsonl(row):
    return {
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": row['Question']},
            {"role": "assistant", "content": row['Answer']}
        ]}

df = pd.read_csv('labor_law_qa.csv')

train_df, test_df = train_test_split(df, test_size=0.20, random_state=42, stratify=df['Source'])

train_data = train_df.apply(format_to_jsonl, axis=1).tolist()
test_data = test_df.apply(format_to_jsonl, axis=1).tolist()

with open('train_data.jsonl', 'w', encoding='utf-8') as f:
    for entry in train_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')

with open('test_data.jsonl', 'w', encoding='utf-8') as f:
    for entry in test_data:
        json.dump(entry, f, ensure_ascii=False)
        f.write('\n')

test_df.to_csv('test_set_ground_truth.csv', index=False, encoding='utf-8-sig')

print(f"Training samples: {len(train_data)} (80%)")
print(f"Test samples: {len(test_data)} (20%)")
print("Files created: train_data.jsonl, test_data.jsonl, test_set_ground_truth.csv")

Training samples: 256 (80%)
Test samples: 65 (20%)
Files created: train_data.jsonl, test_data.jsonl, test_set_ground_truth.csv


# The Baseline Model

It's implemented using BM25, a proven statistical search algorithm. Which works by retrieving the most relevant articles from the cleaned data based on keyword matches.

1. Corpus Building:

* How it works: it iterates through the four cleaned CSV files. And for each row, it concatenated the **Article** column and the **Text** column (e.g., "المادة 77: ما لم يتضمن العقد تعويضاً محدداً مقابل إنهائه...") into a single master list called **CORPUS_OF_LAW**.

* We did it this way to make the retrieved chunk tracable when we do the evaluation step.

2. Tokenization

* How it works: The simple_tokenize function took the raw Arabic text and broke it into a list of standardized words.

* We did this because BM25 cannot process sentences instead, it must compare lists of words, and it removed punctuation, ensuring, for example, that searching for "تعويض" matches ".تعويض"

3. Using BM25Okapi() to Search

* How it works:
  * The model builds an index of the entire dataset to track the frequency of every word.

  * When a query arrives, it matches the user's keywords against this index to calculate a relevance score for every text chunk.

  * Finally, it retrieves the top three highest scoring results and saves it in `model_a_results.csv`.

* We return the top 3 results (instead of 1) to give it the maximum chance of retrieving the correct article.

In [19]:
INPUT_FILES = [
    'boe_cleaned.csv',
    'istitlaa_cleaned.csv',
    'labor_law_faq_cleaned.csv',
    'qiwa_data_cleaned.csv'
]

CORPUS_OF_LAW = []

for file_name in INPUT_FILES:
    try:

        df = pd.read_csv(file_name)

        if 'Text' in df.columns and 'Article' in df.columns:

            corpus_data = df.apply(
                lambda row: f"{row['Article']}: {row['Text']}",
                axis=1
            ).tolist()
            CORPUS_OF_LAW.extend(corpus_data)
        elif 'Text' in df.columns:

            CORPUS_OF_LAW.extend(df['Text'].astype(str).tolist())

    except FileNotFoundError:
        print(f"Warning: File not found: {file_name}")
    except Exception as e:
        print(f"Error processing {file_name}: {e}")

if not CORPUS_OF_LAW:
    raise ValueError("The corpus is empty. Check if the files are correct and contain the 'Text' column.")

print(f"Corpus built successfully with {len(CORPUS_OF_LAW)} documents (articles/chunks).")



def simple_tokenize(text):
  if pd.isna(text):
    return []
  text = str(text).translate(str.maketrans('', '', string.punctuation)).lower()
  return text.split()

tokenized_corpus = [simple_tokenize(doc) for doc in CORPUS_OF_LAW]

bm25 = BM25Okapi(tokenized_corpus)


def run_bm25_baseline(query):
    """Runs a query and retrieves the top 3 most relevant legal chunks."""
    tokenized_query = simple_tokenize(query)

    top_n_documents = bm25.get_top_n(tokenized_query, CORPUS_OF_LAW, n=3)

    return top_n_documents

Corpus built successfully with 248 documents (articles/chunks).


In [17]:
try:
    test_questions_df = pd.read_csv('test_set_ground_truth.csv')
    evaluation_results = []

    for index, row in test_questions_df.iterrows():
        question = row['Question']
        top_docs = run_bm25_baseline(question)

        evaluation_results.append({
            'question': question,
            'top_retrieved_chunk': top_docs[0],
            'top_2_retrieved_chunk': top_docs[1],
            'top_3_retrieved_chunk': top_docs[2],
        })

    pd.DataFrame(evaluation_results).to_csv('model_a_results.csv', index=False)
    print("\nModel A evaluation file (model_a_results.csv) created for 65 test questions.")

except FileNotFoundError:
    print("\n[NOTE]: Cannot run full evaluation loop. Make sure 'test_set_ground_truth.csv' is available.")


Model A evaluation file (model_a_results.csv) created for 65 test questions.
